# 💳 Credit Card Fraud Detection — End-to-End Machine Learning Pipeline

**Objective**: Build and benchmark machine learning models to detect fraudulent credit card transactions in a heavily imbalanced dataset (~0.17% fraud rate).

### Pipeline Overview:
1. **Problem Definition & Business Context**
2. **Data Ingestion & Integrity Checks**
3. **Exploratory Data Analysis (EDA) & Imbalance Dynamics**
4. **Data Preprocessing & Data Leakage Prevention (Stratified Split, Robust Scaling)**
5. **Handling Class Imbalance (SMOTE vs Class Weights)**
6. **Model Training & Comparison (Logistic Regression, Decision Tree, Random Forest, XGBoost, Voting Ensemble)**
7. **Hyperparameter Tuning (RandomizedSearchCV on PR-AUC)**
8. **Evaluation Metrics (Precision, Recall, F1, PR-AUC, ROC-AUC, Confusion Matrix)**
9. **Business Trade-off Analysis (False Positives vs False Negatives)**
10. **Model Serialization for Production Inference**

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))

# Preprocessing & ML libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully!")

## 2. Dataset Loading & Inspection

In [ ]:
data_path = "../data/creditcard.csv"
if not os.path.exists(data_path):
    from data.download_or_generate_data import ensure_dataset
    ensure_dataset("../data")

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Records: {df.duplicated().sum()}")
df.head()

## 3. Exploratory Data Analysis & Class Imbalance

In [ ]:
class_counts = df['Class'].value_counts()
print("Class Distribution:")
print(f"  Legitimate (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)")
print(f"  Fraudulent (1): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.3f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Class', palette=['#2b5c8f', '#e63946'])
plt.yscale('log')
plt.title("Class Distribution (Log Scale)")
plt.ylabel("Count (Log)")
plt.show()

## 4. Stratified Train-Test Split & Robust Feature Scaling

In [ ]:
# 1. Separate Features and Target
X = df.drop(columns=['Class'])
y = df['Class']

# 2. Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Robust Scaling on Time & Amount (Fitted ONLY on training fold)
scaler = RobustScaler()
scale_cols = ['Time', 'Amount']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"Train samples: {X_train.shape[0]:,} (Fraud: {y_train.sum():,})")
print(f"Test samples:  {X_test.shape[0]:,} (Fraud: {y_test.sum():,})")

## 5. Handling Imbalance with SMOTE (Training Data Only)

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Training Set Before SMOTE:", y_train.value_counts().to_dict())
print("Training Set After SMOTE: ", y_train_smote.value_counts().to_dict())

## 6. Model Training & Comparison

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=80, max_depth=10, class_weight='balanced', random_state=42, n_jobs=1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, eval_metric="logloss", random_state=42, n_jobs=1)
}

results = {}
for name, clf in models.items():
    if name == "XGBoost":
        clf.fit(X_train_smote, y_train_smote)
    else:
        clf.fit(X_train_scaled, y_train)
        
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    results[name] = {
        "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1,
        "ROC_AUC": roc_auc, "PR_AUC": pr_auc, "y_prob": y_prob
    }

pd.DataFrame(results).T[['Accuracy', 'Precision', 'Recall', 'F1', 'PR_AUC', 'ROC_AUC']]

## 7. Hyperparameter Tuning on XGBoost

In [ ]:
param_grid = {
    'n_estimators': [80, 120],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
xgb_cv = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=1),
    param_distributions=param_grid,
    n_iter=4,
    scoring='average_precision',
    cv=cv_strat,
    random_state=42,
    n_jobs=1
)
xgb_cv.fit(X_train_smote, y_train_smote)
print("Best Parameters:", xgb_cv.best_params_)
best_xgb = xgb_cv.best_estimator_

## 8. Evaluation: ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['ROC_AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title("ROC Curves")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].legend()

# PR Curve
for name, res in results.items():
    prec_c, rec_c, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(rec_c, prec_c, label=f"{name} (PR-AUC={res['PR_AUC']:.3f})")
axes[1].set_title("Precision-Recall Curves (Minority Class)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
feat_imp = pd.Series(best_xgb.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)
plt.figure(figsize=(9, 4))
feat_imp.plot(kind='barh', color='#2b5c8f')
plt.title("Top 12 Most Discriminative Features for Fraud")
plt.xlabel("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

## 10. Model Persistence

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(best_xgb, "../models/best_model.joblib")
joblib.dump(scaler, "../models/scaler.joblib")
print("Model and Scaler successfully saved in models/ directory!")